# LM from scratch

## BPE Tokenizer

In [1]:
text = """
I’m sorry, but I don’t want to be an emperor. That’s not my business. I don’t want to rule or conquer anyone. I should like to help everyone - if possible - Jew, Gentile - black man - white. We all want to help one another. Human beings are like that. We want to live by each other’s happiness - not by each other’s misery. We don’t want to hate and despise one another. In this world there is room for everyone. And the good earth is rich and can provide for everyone. The way of life can be free and beautiful, but we have lost the way.

Greed has poisoned men’s souls, has barricaded the world with hate, has goose-stepped us into misery and bloodshed. We have developed speed, but we have shut ourselves in. Machinery that gives abundance has left us in want. Our knowledge has made us cynical. Our cleverness, hard and unkind. We think too much and feel too little. More than machinery we need humanity. More than cleverness we need kindness and gentleness. Without these qualities, life will be violent and all will be lost…

The aeroplane and the radio have brought us closer together. The very nature of these inventions cries out for the goodness in men - cries out for universal brotherhood - for the unity of us all. Even now my voice is reaching millions throughout the world - millions of despairing men, women, and little children - victims of a system that makes men torture and imprison innocent people.

To those who can hear me, I say - do not despair. The misery that is now upon us is but the passing of greed - the bitterness of men who fear the way of human progress. The hate of men will pass, and dictators die, and the power they took from the people will return to the people. And so long as men die, liberty will never perish…

Soldiers! don’t give yourselves to brutes - men who despise you - enslave you - who regiment your lives - tell you what to do - what to think and what to feel! Who drill you - diet you - treat you like cattle, use you as cannon fodder. Don’t give yourselves to these unnatural men - machine men with machine minds and machine hearts! You are not machines! You are not cattle! You are men! You have the love of humanity in your hearts! You don’t hate! Only the unloved hate - the unloved and the unnatural! Soldiers! Don’t fight for slavery! Fight for liberty!

In the 17th Chapter of St Luke it is written: “the Kingdom of God is within man” - not one man nor a group of men, but in all men! In you! You, the people have the power - the power to create machines. The power to create happiness! You, the people, have the power to make this life free and beautiful, to make this life a wonderful adventure.

Then - in the name of democracy - let us use that power - let us all unite. Let us fight for a new world - a decent world that will give men a chance to work - that will give youth a future and old age a security. By the promise of these things, brutes have risen to power. But they lie! They do not fulfil that promise. They never will!

Dictators free themselves but they enslave the people! Now let us fight to fulfil that promise! Let us fight to free the world - to do away with national barriers - to do away with greed, with hate and intolerance. Let us fight for a world of reason, a world where science and progress will lead to all men’s happiness. Soldiers! in the name of democracy, let us all unite!
"""

In [2]:
train_text_bytes = text.encode("utf-8")
print(len(train_text_bytes))
list(train_text_bytes)

# so far dictionary has size 255

3409


[10,
 73,
 226,
 128,
 153,
 109,
 32,
 115,
 111,
 114,
 114,
 121,
 44,
 32,
 98,
 117,
 116,
 32,
 73,
 32,
 100,
 111,
 110,
 226,
 128,
 153,
 116,
 32,
 119,
 97,
 110,
 116,
 32,
 116,
 111,
 32,
 98,
 101,
 32,
 97,
 110,
 32,
 101,
 109,
 112,
 101,
 114,
 111,
 114,
 46,
 32,
 84,
 104,
 97,
 116,
 226,
 128,
 153,
 115,
 32,
 110,
 111,
 116,
 32,
 109,
 121,
 32,
 98,
 117,
 115,
 105,
 110,
 101,
 115,
 115,
 46,
 32,
 73,
 32,
 100,
 111,
 110,
 226,
 128,
 153,
 116,
 32,
 119,
 97,
 110,
 116,
 32,
 116,
 111,
 32,
 114,
 117,
 108,
 101,
 32,
 111,
 114,
 32,
 99,
 111,
 110,
 113,
 117,
 101,
 114,
 32,
 97,
 110,
 121,
 111,
 110,
 101,
 46,
 32,
 73,
 32,
 115,
 104,
 111,
 117,
 108,
 100,
 32,
 108,
 105,
 107,
 101,
 32,
 116,
 111,
 32,
 104,
 101,
 108,
 112,
 32,
 101,
 118,
 101,
 114,
 121,
 111,
 110,
 101,
 32,
 45,
 32,
 105,
 102,
 32,
 112,
 111,
 115,
 115,
 105,
 98,
 108,
 101,
 32,
 45,
 32,
 74,
 101,
 119,
 44,
 32,
 71,
 101,
 110,
 116,
 105,
 1

In [3]:
def get_max_occurring(sequence):
    stats: dict[(int, int), int] = {}
    for pair in zip (sequence, sequence[1:]):
        stats[pair] = stats.get(pair, 0) + 1
    
    occurrences = sorted(((v, k) for k, v in stats.items()), reverse=True)
    _, to_be_replaced = max(occurrences)

    return to_be_replaced

In [4]:
# initially, we consider 1 token = 1 byte
sequence = list(train_text_bytes)

to_be_replaced = get_max_occurring(sequence)

to_be_replaced

(101, 32)

In [5]:
def merge(sequence, to_be_replaced, new_token):
    new_sequence = []
    i = 0
    while i < len(sequence):
        if i < len(sequence) - 1 and (sequence[i], sequence[i+1]) == to_be_replaced:
            new_sequence.append(new_token)
            i += 2
        else:
            new_sequence.append(sequence[i])
            i += 1
    return new_sequence


In [6]:
# now we define a new token in our dictionary
new_token = 256

# and substitute the most recurring bytes pair with the new token
new_sequence = merge(sequence, to_be_replaced, new_token)

# the length of the sequence must have decreased accordingly
# assert len(text_bytes) - len(sequence) == count

len(new_sequence)

3280

In [7]:
# now we iterate a bunch

sequence = train_text_bytes
print("initial sequence length: ", len(sequence))

merges = {}
# the number of iterations define the desired size of the dictionary
for i in range(1, 50):
    new_token = 255 + i

    to_be_replaced = get_max_occurring(sequence)

    sequence = merge(sequence, to_be_replaced, new_token)
    print(f"merged pair {to_be_replaced} into new token {new_token}")
    merges[to_be_replaced] = new_token
    print("new sequence lenght: ", len(sequence))

print(merges)


initial sequence length:  3409
merged pair (101, 32) into new token 256
new sequence lenght:  3280
merged pair (116, 104) into new token 257
new sequence lenght:  3207
merged pair (116, 32) into new token 258
new sequence lenght:  3141
merged pair (115, 32) into new token 259
new sequence lenght:  3080
merged pair (100, 32) into new token 260
new sequence lenght:  3033
merged pair (97, 110) into new token 261
new sequence lenght:  2986
merged pair (101, 114) into new token 262
new sequence lenght:  2940
merged pair (111, 32) into new token 263
new sequence lenght:  2901
merged pair (105, 110) into new token 264
new sequence lenght:  2867
merged pair (101, 110) into new token 265
new sequence lenght:  2834
merged pair (46, 32) into new token 266
new sequence lenght:  2803
merged pair (111, 117) into new token 267
new sequence lenght:  2773
merged pair (257, 256) into new token 268
new sequence lenght:  2744
merged pair (45, 32) into new token 269
new sequence lenght:  2715
merged pair (

In [8]:
print("final sequence length: ", len(sequence))
print("initial sequence length: ", len(train_text_bytes))
print(f"compression ratio: {len(train_text_bytes) / len(sequence):.2f}X")

final sequence length:  2139
initial sequence length:  3409
compression ratio: 1.59X


In [9]:
vocab = {i: bytes([i]) for i in range(256)}
for pair, i in merges.items():
    # byte concatenation
    vocab[i] = vocab[pair[0]] + vocab[pair[1]]

vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [10]:
def decode(sequence, vocab):
    text = b""
    for token in sequence:
        text += vocab[token]
    # not all byte sequences are valid utf-8
    # replace invalid ones instead of throwing
    return text.decode("utf-8", errors="replace")

In [11]:
# decoding an invalid utf8 byte sequence fails by returning a special character
decode([128], vocab)

'�'

In [12]:
decoded_bytes = decode(sequence, vocab)

decoded_bytes

'\nI’m sorry, but I don’t want to be an emperor. That’s not my business. I don’t want to rule or conquer anyone. I should like to help everyone - if possible - Jew, Gentile - black man - white. We all want to help one another. Human beings are like that. We want to live by each other’s happiness - not by each other’s misery. We don’t want to hate and despise one another. In this world there is room for everyone. And the good earth is rich and can provide for everyone. The way of life can be free and beautiful, but we have lost the way.\n\nGreed has poisoned men’s souls, has barricaded the world with hate, has goose-stepped us into misery and bloodshed. We have developed speed, but we have shut ourselves in. Machinery that gives abundance has left us in want. Our knowledge has made us cynical. Our cleverness, hard and unkind. We think too much and feel too little. More than machinery we need humanity. More than cleverness we need kindness and gentleness. Without these qualities, life wi

In [13]:
def find_all(text_bytes, token):
    indeces = []
    next = 0
    while True:
        index = text_bytes.find(token, next)
        if index == -1:
            return indeces
        indeces.append(index)
        next = index + len(token)

In [14]:
text_bytes = b"Hello world! lo"
token = b"lo"

find_all(text_bytes, token)


[3, 13]

In [15]:
def encode(text, vocab):
    text_bytes = text.encode('utf-8')
    map = {}
    print(f"text bytes {text_bytes}")
    for i, token in sorted(vocab.items(), reverse=True):
        if token == b"|":
            continue
        indeces = find_all(text_bytes, token)
        if len(indeces) == 0:
            continue
        print(f"looking for token {token}")
        for index in indeces:
            map[index] = (i, token)
            print(f"found at index {index}")
        text_bytes = text_bytes.replace(token, b"|" * len(token))
        print(f"text bytes {text_bytes}")
    
    return [pair[0] for _, pair in sorted(map.items())]

In [16]:
encoded = encode("hello world! I'm a basic tokenizer", vocab)
print(encoded)

decoded = decode(encoded, vocab)
print(decoded)

text bytes b"hello world! I'm a basic tokenizer"
looking for token b'! '
found at index 11
text bytes b"hello world||I'm a basic tokenizer"
looking for token b'or'
found at index 7
text bytes b"hello w||ld||I'm a basic tokenizer"
looking for token b'en'
found at index 28
text bytes b"hello w||ld||I'm a basic tok||izer"
looking for token b'o '
found at index 4
text bytes b"hell||w||ld||I'm a basic tok||izer"
looking for token b'er'
found at index 32
text bytes b"hell||w||ld||I'm a basic tok||iz||"
looking for token b'z'
found at index 31
text bytes b"hell||w||ld||I'm a basic tok||i|||"
looking for token b'w'
found at index 6
text bytes b"hell|||||ld||I'm a basic tok||i|||"
looking for token b't'
found at index 25
text bytes b"hell|||||ld||I'm a basic |ok||i|||"
looking for token b's'
found at index 21
text bytes b"hell|||||ld||I'm a ba|ic |ok||i|||"
looking for token b'o'
found at index 26
text bytes b"hell|||||ld||I'm a ba|ic ||k||i|||"
looking for token b'm'
found at index 15
text byt

In [17]:
def encode_v2(text, merges):
    sequence = text.encode('utf-8')
    while True and len(sequence) > 1:
        # get all pairs
        pairs = set()
        for pair in zip (sequence, sequence[1:]):
            pairs.add(pair)

        # get the pair with the smallest token assigned among the merges
        min_pair = min(pairs, key=lambda p: merges.get(p, float("inf")))
        if min_pair not in merges:
            # nothing more to merge
            break
        token = merges[min_pair]
        sequence = merge(sequence, min_pair, token)
    
    return sequence

In [18]:
encoded = encode_v2("hello world! I'm a basic tokenizer", merges)
print(encoded)

decoded = decode(encoded, vocab)
print(decoded)

[104, 101, 108, 108, 263, 119, 271, 108, 100, 280, 73, 39, 109, 32, 97, 32, 98, 97, 115, 105, 99, 32, 116, 111, 107, 265, 105, 122, 262]
hello world! I'm a basic tokenizer


In [19]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

In [20]:
from lib.tokenizer import Tokenizer

sequence = list(train_text_bytes)

tokenizer = Tokenizer(300)

tokenizer.train(sequence)

tokenizer.vocabulary

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [21]:

encoded = tokenizer.encode("hello world! I'm a basic tokenizer")

decoded = tokenizer.decode(encoded)

decoded

"hello world! I'm a basic tokenizer"

In [22]:

encoded = tokenizer.encode_v2("hello world! I'm a basic tokenizer")

decoded = tokenizer.decode(encoded)

decoded

"hello world! I'm a basic tokenizer"